# 🦜LangChain `create_agent`

**Welcome!**
In the previous notebook, we built a graph manually (nodes, edges, state) using LangGraph. That gave us total control but required a lot of setup.
This notebook introduces **`create_agent`**, a high-level function from the `langchain` module.

###  What is `create_agent`?
It is a **pre-assembled LangGraph**.
Under the hood, `create_agent` automatically:
1.  Creates a `StateGraph`.
2.  Adds a model node and a tool node.
3.  Connects them with the correct edges (loops).
4.  Compiles it into a `CompiledGraph`.

### **📚 What We Will Cover**
1.  **`create_agent`:** Instantiating a pre-built ReAct agent workflow (Model + Tools loop) without needing to define nodes and edges manually.
2.  **Standard Interrupts:** Handling missing info (just like before).
3.  **Middleware:** A way to hook into the agent's brain to modify its behavior (e.g., adding a "Human Approval" step before dangerous tools are run).

![Diagram](./images/10_img.png)

In [1]:
from langchain.agents import create_agent
from langraph_prompts import ACTION_PROMPT_TEMPLATE
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True) # .env values take precedence over existing shell env vars

True

In [2]:
@tool
def get_time_off_balance(user_id: str) -> str:
    '''
    Get the time off balance for a user. 

    Args :
        user_id: A unique string representing the user id

    Returns:
        An integer value representing the time-off balance
    '''
    return 10

@tool
def process_time_off(user_id: str, start_date: str, end_date: str) -> dict:
    '''
    Process time-off request for a user

    Args :
        user_id: A unique string representing the user id
        start_date: A date string in YYYY-MM-DD format for the start date of the time-off request
        end_date: A date string in YYYY-MM-DD format for the end date of the time-off request

    Returns:
        A dictionary with the following keys:
        - status: A string value representing the status of the time-off request
        - message: A string value representing the message of the time-off request
    '''
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    days_requested = end_date - start_date
    time_off_balance = get_time_off_balance.invoke({"user_id": user_id})

    result = {}

    if days_requested.days + 1 > time_off_balance:
        result['status'] = 'error'
        result['message'] = f'Time off request failed. you only have {time_off_balance} days of remaining leaves but requested {days_requested.days} days'

    else:
        time_off_balance -= days_requested.days + 1
        result['status'] = 'success'
        result['message'] = f'Time off request processed successfully for {days_requested.days + 1} days. Your remaining time off balance is {time_off_balance}'


    return result


@tool
def get_additional_info_from_human( message: str) -> str:
    '''
        Raises an interrupt to fetch additional information from the human requesting the action

        Args:
            message: a message string from the AI requesting the user for missing information

        Returns:
            A string value representing the response with additional information from the human

    '''
    interrupt_result = interrupt(message)

    return interrupt_result['user_message']


## 1) Creating the Agent (The Easy Way)

* **Manual Graph (Previous Notebook):** Like building a PC from scratch. You buy the CPU, RAM, and Motherboard, and connect the wires yourself. It's powerful but takes time.
* **`create_agent` (This Notebook):** Like buying a pre-built laptop. You just turn it on. It has the same components inside (LLM, Tools, Memory), but LangChain assembled them for you.

###  Code Context
In the code below:
* `create_agent(...)`: This single function replaces the `StateGraph`, `add_node`, `add_edge`, and `compile` steps.
* It automatically creates a graph where the LLM decides to call tools and loops back until it's done.
* `checkpointer=InMemorySaver()`: We still need this to enable "Pause/Resume" functionality.

📚 **[Docs: create_agent](https://docs.langchain.com/oss/python/langchain/agents)**

![Diagram](./images/11_img.png)

In [23]:
user_id = '1'
llm = init_chat_model('gpt-4o', temperature=0)
tools = [get_time_off_balance, process_time_off, get_additional_info_from_human]

todays_date = datetime.now().strftime('%Y-%m-%d')

system_prompt = ACTION_PROMPT_TEMPLATE.format(todays_date=todays_date, user_id=user_id)

agent = create_agent(
    llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [24]:
user_message = 'I want to take leave'

config = {'configurable': {'thread_id': user_id}}

In [25]:
result = agent.invoke(
    {
        'messages': [{'role': 'user', 'content': user_message}]
    }, 
    config=config
)

In [6]:
result

{'messages': [HumanMessage(content='I want to take leave', additional_kwargs={}, response_metadata={}, id='abf01742-0c3c-4322-ad35-966298cdd947'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 343, 'total_tokens': 378, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_773fccf863', 'id': 'chatcmpl-EbtNjknYs9WXDiHH5KAdGo0hkEz2Y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1653b-0799-726d-ab73-a6192eaca636-0', tool_calls=[{'name': 'get_additional_info_from_human', 'args': {'message': 'Please provide the start and end da

In [26]:
# Readable conversation, including tool calls and results
for message in result["messages"]:
    message.pretty_print()

# Show any pending request for human input
for pause in result.get("__interrupt__", []):
    print(f"\n⏸ Waiting for your input:\n{pause.value}")

================================ Human Message =================================

I want to take leave
================================== Ai Message ==================================
Tool Calls:
  get_additional_info_from_human (call_JArtDIOsIQDG70xo5GeEN4JS)
 Call ID: call_JArtDIOsIQDG70xo5GeEN4JS
  Args:
    message: Please provide the start and end dates for your leave request.

⏸ Waiting for your input:
Please provide the start and end dates for your leave request.


In [27]:
# Save the full result, including messages, tool calls, metadata, and interrupts.
from json_utils import toJson

json_path = toJson(result, "create_agent_result_1.json")
print(f"Saved result to {json_path}")

Saved result to /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-langgraph/.data/create_agent_result_1.json


## 1.1) Resuming the Agent

Just like in the previous notebook, our `get_additional_info_from_human` tool triggered an **Interrupt**. The agent is now paused, waiting for the "End Date".
We use the same `Command(resume=...)` pattern to provide the answer and un-pause execution.

### Code Context
In the code below:
* `agent.invoke(...)`: We pass the `Command` object with the user's answer (`"tomorrow"`).
* The agent takes this answer, pretends the tool returned it, and continues its logic to book the leave.

In [28]:
user_message = 'tomorrow'

result = agent.invoke(Command(resume={'user_message': user_message}), config=config)

result

{'messages': [HumanMessage(content='I want to take leave', additional_kwargs={}, response_metadata={}, id='391fa2f9-f6d9-4975-8523-ddfd361a7328'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 343, 'total_tokens': 373, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_773fccf863', 'id': 'chatcmpl-EbtoG93CIzDk8fLYSynYEmH4EWSo9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a16554-2598-77fc-9033-9afb22f1585c-0', tool_calls=[{'name': 'get_additional_info_from_human', 'args': {'message': 'Please provide the start and end da

In [29]:
# Readable conversation, including tool calls and results
for message in result["messages"]:
    message.pretty_print()

# Show any pending request for human input
for pause in result.get("__interrupt__", []):
    print(f"\n⏸ Waiting for your input:\n{pause.value}")

================================ Human Message =================================

I want to take leave
================================== Ai Message ==================================
Tool Calls:
  get_additional_info_from_human (call_JArtDIOsIQDG70xo5GeEN4JS)
 Call ID: call_JArtDIOsIQDG70xo5GeEN4JS
  Args:
    message: Please provide the start and end dates for your leave request.
================================= Tool Message =================================
Name: get_additional_info_from_human

tomorrow
================================== Ai Message ==================================
Tool Calls:
  get_time_off_balance (call_FPTysITBe6X3pZEpJsgO4cZL)
 Call ID: call_FPTysITBe6X3pZEpJsgO4cZL
  Args:
    user_id: 1
  process_time_off (call_0cKEelhtEpVDNH0XgXX2s9dn)
 Call ID: call_0cKEelhtEpVDNH0XgXX2s9dn
  Args:
    user_id: 1
    start_date: 2026-10-23
    end_date: 2026-10-23
================================= Tool Message =================================
Name: get_time_off_balance

1

In [30]:
# Save the full result, including messages, tool calls, metadata, and interrupts.
from json_utils import toJson

json_path = toJson(result, "create_agent_result_2.json")
print(f"Saved result to {json_path}")

Saved result to /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-langgraph/.data/create_agent_result_2.json


## 2) Middleware (Human-in-the-Loop)

Middleware in LangChain is a new abstraction that lets you hook into and control the agent’s core loop at defined points (before the model is called, as the model is called, and after the model/tool step finishes). It is conceptually similar to HTTP middleware in web frameworks: you can stack multiple middleware components that each inspect or modify state as a request “flows” through the agent.

Each middleware can read and modify the agent state, messages, tools, or model configuration at specific hook points, without changing the agent implementation itself.

In our code, we are adding this "Are you sure?" popup to the **Book Leave** tool to prevent accidental bookings.

### Code Context
In the code below:
* `HumanInTheLoopMiddleware`: This is the component that intercepts tool calls.
* `interrupt_on`: This dictionary defines **which** tools trigger the pause. We map `process_time_off` to specific permissions, ensuring the agent cannot run this specific tool without external approval.

📚 **[Docs: Middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview)**

![Diagram](./images/12_img.png)

In [32]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

In [48]:
agent_with_middleware = create_agent(
    llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "process_time_off": {
                    "allowed_decisions": ["approve", "reject"]
                }
            }
        )
    ]
)

In [49]:
from uuid import uuid4

user_message = "I want to take leave on 17-06-2026"
config = {'configurable': {'thread_id': str(uuid4())}}

In [50]:
result = agent_with_middleware.invoke(
    {"messages": 
     [{"role": "user", "content": user_message}]
    },
    config=config
)
result

# Readable conversation, including tool calls and results
for message in result["messages"]:
    message.pretty_print()

# Show any pending request for human input
for pause in result.get("__interrupt__", []):
    print(f"\n⏸ Waiting for your input:\n{pause.value}")

================================ Human Message =================================

I want to take leave on 17-06-2026
================================== Ai Message ==================================
Tool Calls:
  get_additional_info_from_human (call_eJAmsPwkMQgyHqWwxlxSG6bJ)
 Call ID: call_eJAmsPwkMQgyHqWwxlxSG6bJ
  Args:
    message: Could you please confirm the end date for your leave request? Is it just for one day on 17-06-2026, or does it extend over multiple days?

⏸ Waiting for your input:
Could you please confirm the end date for your leave request? Is it just for one day on 17-06-2026, or does it extend over multiple days?


In [51]:
result = agent_with_middleware.invoke(Command(resume={'user_message': 'single day'}), config=config)

# Readable conversation, including tool calls and results
for message in result["messages"]:
    message.pretty_print()

# Show any pending request for human input
for pause in result.get("__interrupt__", []):
    print(f"\n⏸ Waiting for your input:\n{pause.value}")

================================ Human Message =================================

I want to take leave on 17-06-2026
================================== Ai Message ==================================
Tool Calls:
  get_additional_info_from_human (call_eJAmsPwkMQgyHqWwxlxSG6bJ)
 Call ID: call_eJAmsPwkMQgyHqWwxlxSG6bJ
  Args:
    message: Could you please confirm the end date for your leave request? Is it just for one day on 17-06-2026, or does it extend over multiple days?
================================= Tool Message =================================
Name: get_additional_info_from_human

single day
================================== Ai Message ==================================
Tool Calls:
  get_time_off_balance (call_a4YNMRB8Aklfv08ROjV8MIPk)
 Call ID: call_a4YNMRB8Aklfv08ROjV8MIPk
  Args:
    user_id: 1
  process_time_off (call_2cjtZnWOSJmU2vz9zjA4Ufwb)
 Call ID: call_2cjtZnWOSJmU2vz9zjA4Ufwb
  Args:
    user_id: 1
    start_date: 2026-06-17
    end_date: 2026-06-17

⏸ Waiting for your

In [53]:
result = agent_with_middleware.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)      

# Readable conversation, including tool calls and results
for message in result["messages"]:
    message.pretty_print()

# Show any pending request for human input
for pause in result.get("__interrupt__", []):
    print(f"\n⏸ Waiting for your input:\n{pause.value}")

================================ Human Message =================================

I want to take leave on 17-06-2026
================================== Ai Message ==================================
Tool Calls:
  get_additional_info_from_human (call_eJAmsPwkMQgyHqWwxlxSG6bJ)
 Call ID: call_eJAmsPwkMQgyHqWwxlxSG6bJ
  Args:
    message: Could you please confirm the end date for your leave request? Is it just for one day on 17-06-2026, or does it extend over multiple days?
================================= Tool Message =================================
Name: get_additional_info_from_human

single day
================================== Ai Message ==================================
Tool Calls:
  get_time_off_balance (call_a4YNMRB8Aklfv08ROjV8MIPk)
 Call ID: call_a4YNMRB8Aklfv08ROjV8MIPk
  Args:
    user_id: 1
  process_time_off (call_2cjtZnWOSJmU2vz9zjA4Ufwb)
 Call ID: call_2cjtZnWOSJmU2vz9zjA4Ufwb
  Args:
    user_id: 1
    start_date: 2026-06-17
    end_date: 2026-06-17
===================